<a href="https://colab.research.google.com/github/Slh62804/medical-rag-assistant/blob/main/Natural_Language_Processing_with_Generative_AI_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1><center></center></h1>
<h1><center><font size=10>Natural Language Processing with Generative AI Project</center></font></h1>
<h1><center>Medical Assistant</center></h1>

##**Business Context**

The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

##**Objective**

As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to understand information overload, apply AI techniques to streamline decision-making, analyze its impact on diagnostics and patient outcomes, evaluate its potential to standardize care practices, and create a functional prototype demonstrating its feasibility and effectiveness.

**Questions to Answer**

1.  What is the protocol for managing sepsis in a critical care unit?


2.  What are the common symptoms of appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?


3.  What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?


4.  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?


5.  What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

##**Data Dictionary**


The Merck Manuals are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899 when Merck & Co. was still a subsidiary of the German company Merck.

The manual is a PDF with over 4,000 pages divided into 23 sections.

**Installing and Importing Necessary Libraries and Dependencies**

In [ ]:
!CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install llama-cpp-python==0.1.85 --force-reinstall --no-cache-dir -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 42.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 285.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 254.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 296.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
chromadb 0.5.5 requires numpy<2.0.0,>=1.22.5, but you have numpy 2.5.2 which is incompatible.
numba 0.61.2 requires numpy<2.3,>=1.24, but you have numpy 2.5.2 which is incompatible.
langgraph-sdk 0.4.2 requires langchain-core<2,>=1.4.0, but you have langchain-core 0.3.86 which is incompatible.
gradio 6.24.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingfac

In [ ]:
!pip install -q \
    huggingface_hub>=0.25.0 \
    pandas>=2.2.2 \
    tiktoken \
    pymupdf==1.25.1 \
    langchain \
    langchain-community \
    langchain-text-splitters==0.3.6 \
    chromadb==0.5.5 \
    sentence-transformers==3.2.0 \
    numpy

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ml-dtypes 0.6.0 requires numpy>=2.0.0, but you have numpy 1.26.4 which is incompatible.
ml-dtypes 0.6.0 requires numpy>=2.1.0; python_version >= "3.13", but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.52.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
gradio 6.24.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
xarray-einstats 0.11.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.14.0.94 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
dm-tree 0.1.10 requires numpy>=2.1.0; python_version >= "3.13", but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.1 re

In [ ]:
#Libraries for processing dataframes,text
import json,os
import tiktoken
import pandas as pd

#Libraries for Loading Data, Chunking, Embedding, and Vector Databases
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma

from huggingface_hub import hf_hub_download
from llama_cpp import Llama

##Question Answering using LLM

###Downloading and Loading the model

In [ ]:
# Download the Mistral 7B Instruct v0.2 GGUF model from Hugging Face.
# Q4_K_M provides a good balance between model size and response quality.

model_name = "TheBloke/Mistral-7B-Instruct-v0.2-GGUF"
model_file = "mistral-7b-instruct-v0.2.Q4_K_M.gguf"

model_path = hf_hub_download(
    repo_id=model_name,
    filename=model_file
)

print("Model downloaded successfully.")
print("Model path:", model_path)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


mistral-7b-instruct-v0.2.Q4_K_M.gguf:   0%|          | 0.00/4.37G [00:00<?, ?B/s]

Model downloaded successfully.
Model path: /root/.cache/huggingface/hub/models--TheBloke--Mistral-7B-Instruct-v0.2-GGUF/snapshots/3a6fbf4a41a1d52e415a4958cde6856d34b2db93/mistral-7b-instruct-v0.2.Q4_K_M.gguf


In [ ]:
# Load the downloaded Mistral model using llama.cpp.
# n_ctx defines the context window.
# n_gpu_layers=-1 attempts to offload the model layers to the GPU.

llm = Llama(
    model_path=model_path,
    n_ctx=4096,
    n_gpu_layers=-1,
    verbose=True
)

print("Mistral 7B Instruct model loaded successfully.")

Mistral 7B Instruct model loaded successfully.


AVX = 1 | AVX2 = 1 | AVX512 = 0 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | FMA = 1 | NEON = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 1 | SSE3 = 1 | SSSE3 = 1 | VSX = 0 | 


###**Response**

In [ ]:
def response(query,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    model_output = llm(
      prompt=query,
      max_tokens=max_tokens,
      temperature=temperature,
      top_p=top_p,
      top_k=top_k
    )

    return model_output['choices'][0]['text']

`  Query 1: What is the protocol for managing sepsis in a critical care unit?`

In [ ]:
query_1 = """[INST]
What is the protocol for managing sepsis in a critical care unit?
[/INST]"""

response_1 = response(
    query_1,
    max_tokens=300,
    temperature=0,
    top_p=0.95,
    top_k=50
)

print(response_1)

 Sepsis is a life-threatening condition caused by the body's response to an infection, and managing it effectively in a critical care unit requires a multidisciplinary approach. Here are some key steps in the protocol for managing sepsis in a critical care unit:
1. Early recognition and diagnosis: Identify sepsis early by recognizing the signs and symptoms, such as fever or hypothermia, tachycardia or bradycardia, altered mental status, and lactic acidosis. Use the Sequential Organ Failure Assessment (SOFA) score to assess organ dysfunction.
2. Resuscitation: Provide fluid resuscitation to maintain adequate tissue perfusion and oxygenation. Aim for a mean arterial pressure of 65 mmHg or higher and a central venous oxygen saturation of 70% or higher.
3. Antibiotics: Administer broad-spectrum antibiotics as soon as possible based on culture results, clinical suspicion, and local guidelines.
4. Source control: Identify and address the source of infection, such as drainage of abscesses or 

**Observation:**

The baseline LLM outlines sepsis management—stabilization, fluids, antimicrobials, hemodynamic support, and monitoring. However, since it is not yet linked to the Merck Manuals, the response cannot be traced to the designated source. It should be treated only as a preliminary benchmark for later RAG-based outputs.

`Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?`

In [ ]:
query_2 = """[INST]
What are the common symptoms for appendicitis, and can it be cured via medicine?
If not, what surgical procedure should be followed to treat it?
[/INST]"""

response_2 = response(
    query_2,
    max_tokens=300,
    temperature=0,
    top_p=0.95,
    top_k=50
)

print(response_2)

Llama.generate: prefix-match hit


 Appendicitis is a medical condition characterized by inflammation of the appendix, a small tube-shaped organ located in the lower right side of the abdomen. The following are common symptoms of appendicitis:
1. Abdominal pain, usually starting around the navel and then shifting to the lower right side
2. Loss of appetite
3. Nausea and vomiting
4. Fever
5. Diarrhea or constipation
6. Abdominal swelling
7. Inability to pass gas or have a bowel movement
8. Feeling bloated or full
9. Pain upon walking, coughing, or moving
10. Anxiety and restlessness
Appendicitis cannot be cured via medicine alone as the inflammation can cause the appendix to rupture, leading to peritonitis, a serious infection of the abdominal cavity. If left untreated, it can be life-threatening. Therefore, if you suspect appendicitis, it is essential to seek medical attention immediately.
The standard treatment for appendicitis is surgical removal of the appendix, known as an appendectomy. This procedure can be perform

**Observation:**

The LLM should identify common manifestations of appendicitis and discuss medical management and appendectomy. The response demonstrates that a general-purpose LLM can provide useful medical information, but it does not provide evidence that the information is derived from the Merck Manuals. This limitation motivates the use of RAG

`Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?`

In [ ]:
query_3 = """[INST]
What are the effective treatments or solutions for addressing sudden patchy hair loss,
commonly seen as localized bald spots on the scalp, and what could be the possible
causes behind it?
[/INST]"""

response_3 = response(
    query_3,
    max_tokens=350,
    temperature=0,
    top_p=0.95,
    top_k=50
)

print(response_3)

Llama.generate: prefix-match hit


 Sudden patchy hair loss, also known as alopecia areata, is an autoimmune condition that causes hair loss in small patches on the scalp, beard, or other areas of the body. The exact cause of alopecia areata is not known, but it's believed to be related to a problem with the immune system. Here are some effective treatments and solutions for addressing sudden patchy hair loss:
1. Topical corticosteroids: These are anti-inflammatory medications that can help reduce inflammation and promote hair growth. They can be applied directly to the affected area in the form of creams, lotions, or solutions.
2. Injections of corticosteroids: In some cases, corticosteroids may be injected directly into the bald spots to promote hair growth. This treatment is usually reserved for severe cases of alopecia areata.
3. Minoxidil: This medication is applied topically and has been shown to help slow down hair loss and promote new hair growth in some people with alopecia areata.
4. DHT blockers: Dihydrotesto

**Observation:**

 The response is expected to discuss possible causes of localized hair loss, including alopecia areata and other differential diagnoses, together with treatment approaches. However, the baseline model may provide a broad answer without clearly distinguishing which information is supported by the project's reference manual.

`Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?`

In [ ]:
query_4 = """[INST]
What treatments are recommended for a person who has sustained a physical injury
to brain tissue, resulting in temporary or permanent impairment of brain function?
[/INST]"""

response_4 = response(
    query_4,
    max_tokens=350,
    temperature=0,
    top_p=0.95,
    top_k=50
)

print(response_4)

Llama.generate: prefix-match hit


 A person who has sustained a physical injury to the brain, also known as a traumatic brain injury (TBI), may require various treatments depending on the severity and location of the injury. Here are some common treatments recommended for TBIs:
1. Medical Management: This includes addressing any life-threatening conditions such as airway obstruction, breathing problems, or excessive bleeding.
2. Surgical Intervention: In some cases, surgery may be necessary to remove hematomas (clots) or decompress the skull to relieve pressure on the brain.
3. Rehabilitation: Rehabilitation is an essential part of treatment for TBIs. It may include physical therapy to help with mobility and strength, occupational therapy to help with daily living activities, speech therapy to improve communication skills, and cognitive rehabilitation to address memory, attention, and problem-solving abilities.
4. Medications: Depending on the symptoms, medications may be prescribed to manage conditions such as seizure

**Observation:**

The model provides a general description of traumatic brain injury management, potentially including stabilization, neurological assessment, management of intracranial complications and rehabilitation. The main limitation is again the lack of source grounding and traceability.

`Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?`

In [ ]:
query_5 = """[INST]
What are the necessary precautions and treatment steps for a person who has fractured
their leg during a hiking trip, and what should be considered for their care and recovery?
[/INST]"""

response_5 = response(
    query_5,
    max_tokens=350,
    temperature=0,
    top_p=0.95,
    top_k=50
)

print(response_5)

Llama.generate: prefix-match hit


 If a person has fractured their leg during a hiking trip, it is important that they take the following necessary precautions and seek medical attention as soon as possible:
1. Stay calm and try to prevent further injury: Encourage the person to remain still and avoid moving the injured leg as much as possible to prevent further damage or discomfort.
2. Apply ice: Apply an ice pack to the injured area for 15-20 minutes at a time, several times a day, to help reduce swelling and pain.
3. Elevate the leg: Try to keep the injured leg elevated above heart level as much as possible to help reduce swelling and promote blood flow.
4. Immobilize the leg: Use a splint, sling, or brace to immobilize the leg and prevent any movement that could cause further damage.
5. Seek medical attention: It is important to seek medical attention as soon as possible for proper diagnosis and treatment of the fracture. The healthcare professional may recommend immobilization with a cast or brace, pain management

**Observation:**

The model should provide information about immobilization, emergency assessment, fracture management, possible complications and recovery. Since this is an emergency/injury scenario, the response should emphasize professional medical assessment. The baseline answer does not provide a reliable source trail to the Merck Manuals.

##**Question Answering using LLM with Prompt Engineering**

For this section, we need at least five combinations.

Rather than manually creating five separate functions, create one reusable function and five parameter configurations

In [ ]:
def generate_prompt_response(
    question,
    system_instruction,
    max_tokens=300,
    temperature=0,
    top_p=0.95,
    top_k=50
):

    prompt = f"""[INST]
{system_instruction}

Question:
{question}
[/INST]"""

    model_output = llm(
        prompt=prompt,
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k
    )

    return model_output["choices"][0]["text"].strip()

###**Define 5 Combinations**

In [ ]:
prompt_configurations = [

    {
        "name": "Configuration 1 - Basic",
        "instruction": """
Answer the medical question clearly and concisely.
Provide the important symptoms, causes and treatments where applicable.
""",
        "temperature": 0,
        "top_p": 0.95,
        "top_k": 50,
        "max_tokens": 300
    },

    {
        "name": "Configuration 2 - Structured",
        "instruction": """
Answer the medical question using the following structure:
1. Key symptoms/findings
2. Possible causes or diagnosis
3. Treatment/management
4. Important precautions
Keep the answer concise and factual.
""",
        "temperature": 0.2,
        "top_p": 0.9,
        "top_k": 40,
        "max_tokens": 350
    },

    {
        "name": "Configuration 3 - Clinical",
        "instruction": """
Act as a medical information assistant.
Provide a clinically organized answer.
Clearly distinguish symptoms, diagnosis, treatment and complications.
Avoid unsupported claims.
""",
        "temperature": 0.2,
        "top_p": 0.85,
        "top_k": 40,
        "max_tokens": 400
    },

    {
        "name": "Configuration 4 - Conservative",
        "instruction": """
Provide a cautious, evidence-oriented medical response.
Do not invent information.
If there is uncertainty, explicitly mention it.
Focus on clinically important information.
""",
        "temperature": 0,
        "top_p": 0.8,
        "top_k": 30,
        "max_tokens": 400
    },

    {
        "name": "Configuration 5 - Detailed",
        "instruction": """
Provide a comprehensive medical response covering:
symptoms, possible causes, diagnosis, treatment, complications,
precautions and relevant emergency considerations.
Use clear headings and avoid unnecessary repetition.
""",
        "temperature": 0.3,
        "top_p": 0.9,
        "top_k": 50,
        "max_tokens": 500
    }
]

`Query 1: What is the protocol for managing sepsis in a critical care unit? (With all 5 combinations)`

In [ ]:
for config in prompt_configurations:

    print("=" * 100)
    print(config["name"])
    print("=" * 100)

    answer = generate_prompt_response(
        question=query_1.replace("[INST]", "").replace("[/INST]", ""),
        system_instruction=config["instruction"],
        max_tokens=config["max_tokens"],
        temperature=config["temperature"],
        top_p=config["top_p"],
        top_k=config["top_k"]
    )

    print(answer)

Configuration 1 - Basic


Llama.generate: prefix-match hit


Sepsis is a life-threatening condition caused by the body's response to an infection. In a critical care unit, managing sepsis involves the following steps:

1. Recognition and early identification: Identify patients with suspected infection and assess for signs of sepsis, such as fever or hypothermia, tachycardia or bradycardia, respiratory distress, altered mental status, and lactic acidosis.
2. Immediate fluid resuscitation: Administer intravenous fluids to maintain adequate blood pressure and organ perfusion. The goal is to achieve a mean arterial pressure (MAP) of 65 mmHg or higher.
3. Antibiotics: Start broad-spectrum antibiotics as soon as possible, based on the suspected infection site and microbiological cultures if available.
4. Source control: Identify and address the source of infection, such as removing an infected catheter or draining an abscess.
5. Vasopressors: If fluid resuscitation fails to maintain adequate blood pressure, administer vasopressors to support blood pre

Llama.generate: prefix-match hit


1. Key symptoms/findings: Sepsis is a life-threatening condition caused by the body's response to infection. Symptoms may include fever, chills, rapid heart rate, rapid breathing, low blood pressure, confusion or disorientation, and decreased urine output. In severe cases, septic shock can occur, leading to organ failure.
2. Possible causes or diagnosis: Sepsis is usually caused by bacterial infections, most commonly from the lungs (pneumonia), urinary tract, or abdomen. Diagnosis is based on clinical signs and laboratory findings, such as high white blood cell count, elevated lactate levels, and low oxygen levels in the blood.
3. Treatment/management: Early recognition and prompt treatment are crucial for improving outcomes. This includes administering antibiotics to address the underlying infection, providing intravenous fluids to maintain adequate blood pressure, and providing oxygen support if necessary. In some cases, vasopressors may be needed to maintain blood pressure.
4. Impor

...

Sepsis is a life-threatening condition caused by the body's response to an infection. In a critical care unit, managing sepsis requires a swift and coordinated approach to prevent progression to septic shock. Here's a clinically organized answer:

1. Recognition of symptoms:
   * Fever or hypothermia
   * Tachycardia (heart rate >90 bpm) or bradycardia (<60 bpm)
   * Respiratory distress (tachypnea, dyspnea, or altered respiratory status)
   * Altered mental status
   * Decreased urine output
   * Hematologic abnormalities (leukocytosis or leukopenia, thrombocytopenia)
   * Lactic acidosis

2. Diagnosis:
   * Suspected infection and positive blood culture or other microbiological evidence
   * Clinical assessment using Sequential [Sepsis-related] Organ Failure Assessment (SOFA) score

3. Immediate treatment:
   * Oxygen therapy to maintain adequate oxygenation
   * Intravenous fluids to maintain adequate blood pressure and organ perfusion
   * Broad-spectrum antibiotics based on suspec

Llama.generate: prefix-match hit


Sepsis is a life-threatening condition caused by the body's response to infection. In a critical care unit, managing sepsis involves a swift and coordinated approach to address the underlying infection, support organ function, and prevent complications. Here are some key steps in sepsis management based on current clinical guidelines:

1. Early recognition and diagnosis: Recognize sepsis early by assessing for signs of infection and systemic inflammatory response. Use validated scoring systems like the Sequential Organ Failure Assessment (SOFA) or Quick Sequential [Sepsis-related] Organ Failure Assessment (qSOFA) to help identify patients at risk.

2. Fluid resuscitation: Aim for adequate tissue perfusion by administering intravenous fluids to maintain mean arterial pressure ≥65 mmHg and central venous oxygen saturation >70%. Use crystalloids initially, followed by colloids or blood products if needed.

3. Antibiotics: Administer broad-spectrum antibiotics as soon as possible based on 

Llama.generate: prefix-match hit


Title: Sepsis Management Protocol in a Critical Care Unit

1. Symptoms:
Sepsis is a life-threatening condition caused by the body's response to an infection, which can lead to tissue damage and organ failure. Common symptoms include fever or hypothermia, rapid heart rate, rapid breathing, confusion or disorientation, low blood pressure, and decreased urine output. In severe cases, septic shock may occur, leading to further complications such as multi-organ failure.

2. Possible Causes:
Sepsis can be caused by various types of infections, including bacterial, viral, fungal, or parasitic infections. The most common sources are respiratory tract infections (pneumonia), urinary tract infections, and intra-abdominal infections.

3. Diagnosis:
The diagnosis of sepsis is based on clinical evaluation, laboratory findings, and imaging studies. The Sequential [Sepsis-related] Organ Failure Assessment (SOFA) score can help identify patients at risk for developing sepsis. A positive blood culture 

`Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?(With all 5 combinations)`

In [ ]:
for config in prompt_configurations:

    print("=" * 100)
    print(config["name"])
    print("=" * 100)

    answer = generate_prompt_response(
        question=query_2.replace("[INST]", "").replace("[/INST]", ""),
        system_instruction=config["instruction"],
        max_tokens=config["max_tokens"],
        temperature=config["temperature"],
        top_p=config["top_p"],
        top_k=config["top_k"]
    )

    print(answer)

Configuration 1 - Basic


Llama.generate: prefix-match hit


Appendicitis is a medical condition characterized by inflammation of the appendix, a small pouch located in the lower right side of the abdomen. Common symptoms include:

1. Sudden and persistent pain in the lower right abdomen, often starting as a mild discomfort that gradually worsens over hours.
2. Loss of appetite and feeling sick to your stomach (nausea).
3. Vomiting.
4. Fever, especially in higher temperatures (above 101 degrees Fahrenheit or 38 degrees Celsius).
5. Constipation or diarrhea.
6. Abdominal swelling and tenderness.
7. Inability to pass gas or have a bowel movement.

Appendicitis cannot be cured with medicine alone, as the inflammation can lead to the appendix rupturing if left untreated. If diagnosed early, antibiotics may help prevent complications during surgery. However, surgical removal of the appendix (appendectomy) is the standard treatment for appendicitis. This procedure can be performed laparoscopically or open, depending on the severity and individual circ

Llama.generate: prefix-match hit


1. Key symptoms/findings:
- Sudden onset of pain in the lower right abdomen, often starting as a mild discomfort that gradually worsens over hours.
- Loss of appetite and feeling sick to your stomach (nausea).
- Vomiting.
- Fever and chills.
- Pain worsening with movement or pressure on the abdomen.
- Constipation or diarrhea.

2. Possible causes or diagnosis:
Appendicitis is most commonly caused by a blockage in the small, hollow organ located in the lower right side of the abdomen called the appendix. The blockage can be due to fecal matter, lymph, or mucus. If left untreated, the appendix can rupture, leading to peritonitis, a serious inflammation of the abdominal cavity. Diagnosis is typically made through physical examination and imaging tests such as ultrasound or CT scan.

3. Treatment/management:
Appendicitis cannot be cured with medicine alone. The standard treatment for appendicitis is surgical removal of the appendix, known as an appendectomy. This procedure can be performed

...

Appendicitis is a medical condition characterized by inflammation of the appendix, a small pouch located in the lower right side of the abdomen. The following are common symptoms of appendicitis:

1. Abdominal pain: This is usually the first and most prominent symptom. The pain may begin as a vague discomfort in the middle of the abdomen, but it soon localizes to the right lower quadrant. The pain worsens with movement or pressure on the area.
2. Loss of appetite: People with appendicitis often lose their appetite due to the abdominal pain and nausea.
3. Nausea and vomiting: Vomiting is a common symptom, especially in advanced cases.
4. Fever: A low-grade fever may be present.
5. Constipation or diarrhea: Both can occur, but constipation is more common.
6. Rebound tenderness: This is a sign of inflammation and is elicited by gently pressing on the abdomen and then releasing the pressure.
7. Guarding: This is the involuntary contraction of the abdominal muscles in response to pain.
8. R

Llama.generate: prefix-match hit


Appendicitis is a medical condition characterized by inflammation of the appendix, a small tube-shaped organ located in the lower right abdomen. The symptoms of appendicitis can vary but typically include:

1. Abdominal pain, usually starting around the navel and then shifting to the right lower quadrant.
2. Loss of appetite.
3. Nausea and vomiting.
4. Fever.
5. Diarrhea or constipation.
6. Abdominal swelling and tenderness.
7. Pain upon walking or moving around.
8. Inability to pass gas or have a bowel movement.

It is important to note that these symptoms can also be present in other conditions, so a proper diagnosis should always be made by a healthcare professional.

Regarding the treatment for appendicitis, it is generally not curable with medicine alone. The standard treatment is surgical removal of the appendix, known as an appendectomy. This procedure is typically performed laparoscopically, which involves making small incisions and using a camera to guide the surgeon. Open app

Llama.generate: prefix-match hit


**Appendicitis:** A Medical Condition Requiring Prompt Attention

**Symptoms:**
Appendicitis is an inflammatory condition of the appendix, a small tube-like structure attached to the large intestine. The symptoms may include:

1. Sudden and persistent pain in the lower right abdomen
2. Loss of appetite
3. Nausea and vomiting
4. Fever (often low-grade)
5. Abdominal swelling
6. Constipation or diarrhea
7. Inability to pass gas or have a bowel movement
8. Pain that worsens when walking, coughing, or taking deep breaths

**Possible Causes:**
The exact cause of appendicitis is not always clear, but it's believed to be due to an obstruction in the appendix, most commonly by fecal matter. Other possible causes include:

1. Infection from bacteria such as E. coli or Streptococcus
2. Enlarged lymph nodes
3. Tumors or cysts
4. Foreign objects
5. Inflammatory bowel disease

**Diagnosis:**
To diagnose appendicitis, a healthcare professional will consider the patient's symptoms and medical history,

`Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?(With all 5 combinations)`

In [ ]:
for config in prompt_configurations:

    print("=" * 100)
    print(config["name"])
    print("=" * 100)

    answer = generate_prompt_response(
        question=query_3.replace("[INST]", "").replace("[/INST]", ""),
        system_instruction=config["instruction"],
        max_tokens=config["max_tokens"],
        temperature=config["temperature"],
        top_p=config["top_p"],
        top_k=config["top_k"]
    )

    print(answer)

Configuration 1 - Basic


Llama.generate: prefix-match hit


Sudden patchy hair loss, also known as alopecia areata, is an autoimmune condition that causes hair loss in small patches on the scalp. The exact cause of alopecia areata is unknown, but it's believed to be related to an issue with the immune system attacking hair follicles.

Effective treatments for addressing sudden patchy hair loss include:

1. Corticosteroids: These medications can help reduce inflammation and suppress the immune response that causes hair loss. They can be applied topically or taken orally.
2. Immunotherapy: This involves using substances that stimulate an immune response to promote hair regrowth. One common immunotherapy treatment is called diphenylcyclopropenone (DPCP).
3. Minoxidil: This medication can help slow down hair loss and promote new hair growth. It's applied topically.
4. Nutritional supplements: Certain vitamins and minerals, such as biotin and zinc, may help support healthy hair growth.
5. Stress management: Managing stress through techniques like me

Llama.generate: prefix-match hit


1. Key symptoms/findings: Sudden patchy hair loss is characterized by the appearance of localized bald spots on the scalp. The hair in these areas may come out easily when pulled or brushed, leaving small, round patches of exposed skin.
2. Possible causes or diagnosis: There are several possible causes for sudden patchy hair loss, including:
   a. Alopecia Areata: An autoimmune condition that causes hair loss in circular patches on the scalp, beard, or other areas of the body.
   b. Telogen Effluvium: A type of hair loss where large numbers of hairs enter the resting phase (telogen) and fall out prematurely. This can be caused by stress, illness, or certain medications.
   c. Traction Alopecia: Hair loss due to excessive pulling or tension on the hair, often seen in people who wear tight braids or ponytails.
   d. Nutritional deficiencies: Lack of essential nutrients such as iron, zinc, or biotin can lead to hair loss.
3. Treatment/management: The treatment for sudden patchy hair loss 

...

I. Introduction
Sudden patchy hair loss, also known as alopecia areata, is a common autoimmune disorder that results in the sudden appearance of smooth, round, or patchy bald spots on the scalp. This condition can affect people of all ages and genders. In this response, we will discuss the possible causes, symptoms, diagnosis, treatments, and complications of alopecia areata.

II. Causes
The exact cause of alopecia areata is unknown, but it's believed to be an autoimmune disorder where the body's immune system attacks hair follicles, leading to hair loss. Some possible triggers include stress, illness, hormonal imbalances, and certain medications.

III. Symptoms
The primary symptom of alopecia areata is the sudden appearance of smooth, round, or patchy bald spots on the scalp. However, it can also affect other areas of the body such as the beard, eyebrows, or eyelashes. In some cases, complete hair loss (alopecia totalis) or total body hair loss (alopecia universalis) may occur.

IV. D

Llama.generate: prefix-match hit


I. Possible Causes of Sudden Patchy Hair Loss (Alopecia Areata):
1. Autoimmune disorder: The immune system mistakenly attacks hair follicles, causing hair loss.
2. Stress: Emotional or physical stress can trigger the onset of alopecia areata.
3. Genetics: There is a higher risk of developing alopecia areata if there is a family history of the condition.
4. Viral infections: Certain viruses may contribute to hair loss.
5. Other health conditions: Thyroid disease, lupus, or other autoimmune disorders can cause patchy hair loss.
6. Medications: Some drugs, such as certain antidepressants and chemotherapy medications, can lead to hair loss.
7. Trauma: Physical trauma to the scalp, such as burns or injury, may result in patchy hair loss.

II. Effective Treatments for Patchy Hair Loss (Alopecia Areata):
1. Topical treatments: Corticosteroids applied directly to the affected area can help stimulate hair growth. Minoxidil is another topical treatment that has been shown to be effective in some

Llama.generate: prefix-match hit


Title: Sudden Patchy Hair Loss: Causes, Diagnosis, Treatment, Complications, and Precautions

1. Symptoms:
Sudden patchy hair loss, also known as alopecia areata, is characterized by round or oval bald spots on the scalp that may appear suddenly. The affected areas typically have normal-looking skin with fine hairs or no visible hairs at all. In some cases, there might be itching or burning sensations before the hair falls out.

2. Possible Causes:
Alopecia areata is an autoimmune disorder where the body's immune system attacks hair follicles, leading to hair loss. Other possible causes include stress, genetics, vitamin deficiencies (especially iron or biotin), hormonal imbalances, and certain medications. Infections such as ringworm can also cause patchy hair loss.

3. Diagnosis:
A healthcare professional will typically diagnose alopecia areata based on the appearance of the bald spots and a thorough medical history. They may perform blood tests to rule out other conditions that could

**Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?(With all 5 combinations)**

In [ ]:
for config in prompt_configurations:

    print("=" * 100)
    print(config["name"])
    print("=" * 100)

    answer = generate_prompt_response(
        question=query_4.replace("[INST]", "").replace("[/INST]", ""),
        system_instruction=config["instruction"],
        max_tokens=config["max_tokens"],
        temperature=config["temperature"],
        top_p=config["top_p"],
        top_k=config["top_k"]
    )

    print(answer)

Configuration 1 - Basic


Llama.generate: prefix-match hit


Treatment for a brain injury depends on the severity and location of the injury. Here are some common treatments:

1. Medical Management: This includes controlling any bleeding, managing seizures, and addressing any life-threatening conditions.
2. Rehabilitation: Physical therapy, occupational therapy, speech therapy, and cognitive rehabilitation can help improve function and reduce symptoms.
3. Medications: Various medications may be prescribed to manage symptoms such as pain, swelling, seizures, or depression.
4. Surgery: In some cases, surgery may be necessary to remove hematomas (clots) or repair skull fractures.
5. Assistive Devices: Devices like wheelchairs, walkers, or communication aids can help individuals with brain injuries function better in their daily lives.
6. Lifestyle Modifications: Making lifestyle changes such as getting enough rest, eating well, and avoiding alcohol and drugs can promote healing and improve overall health.
7. Support Groups: Joining support groups c

Llama.generate: prefix-match hit


1. Key symptoms/findings: A person with a brain injury may experience various symptoms such as headache, dizziness, loss of consciousness, memory problems, difficulty speaking or understanding speech, and changes in mood or behavior. Seizures, paralysis, and coma are also possible depending on the severity of the injury.
2. Possible causes or diagnosis: A brain injury can be caused by various factors including trauma (e.g., car accidents, falls), infections (e.g., meningitis, encephalitis), tumors, or stroke. Brain injuries are typically diagnosed through a combination of physical examination, imaging studies (such as CT or MRI scans), and neurological assessments.
3. Treatment/management: The treatment for a brain injury depends on the severity and cause of the injury. For mild to moderate brain injuries, rest, rehabilitation, and medication to manage symptoms may be sufficient. Severe brain injuries may require surgery, intensive care unit (ICU) admission, and long-term rehabilitatio

...

I. Introduction
- Brain injuries can result in various degrees of impairment to brain function, ranging from mild concussions to severe traumatic brain injuries (TBI).
- Treatment for brain injuries depends on the severity and specific symptoms present.

II. Symptoms
- Common symptoms of brain injuries include: headache, dizziness, memory loss, difficulty speaking or understanding speech, weakness or paralysis, and changes in mood or behavior.

III. Diagnosis
- Brain injuries are typically diagnosed through a combination of medical history, physical examination, imaging studies (such as CT or MRI scans), and neurological assessments.

IV. Treatment
1. Acute Care:
   - In the acute phase, treatment may include: managing increased intracranial pressure, preventing infection, controlling seizures, and providing supportive care to maintain vital functions.

2. Rehabilitation:
   - Rehabilitation is an essential component of treatment for brain injuries. It may involve physical therapy, occ

Llama.generate: prefix-match hit


Based on current medical knowledge, the treatment for a person with a brain injury depends on the severity and specific location of the injury. Here are some common interventions:

1. Initial Management: This includes addressing any life-threatening conditions such as airway obstruction, breathing difficulties, or excessive bleeding. In some cases, surgery may be required to remove hematomas (clots) or repair skull fractures.

2. Supportive Care: Patients with brain injuries often require close monitoring in an intensive care unit. They may need assistance with breathing, feeding, and other basic functions. Rehabilitation therapies such as physical therapy, occupational therapy, and speech therapy may also be initiated early to help prevent complications and promote recovery.

3. Medications: Various medications can be used to manage symptoms associated with brain injuries, including pain relievers, anti-seizure drugs, and medications to reduce intracranial pressure.

4. Nutritional Su

Llama.generate: prefix-match hit


Title: Brain Injury: Symptoms, Causes, Diagnosis, Treatment, Complications, Precautions, and Emergency Considerations

1. Symptoms of Brain Injury:
- Headache or a feeling of pressure in the head.
- Temporary loss of consciousness or disorientation.
- Confusion or memory problems.
- Dizziness or loss of balance.
- Nausea or vomiting.
- Slurred speech.
- Changes in mood, behavior, or personality.
- Seizures.
- Paralysis or weakness in limbs.
- Sensory impairments such as vision or hearing changes.

2. Possible Causes of Brain Injury:
- Trauma (car accidents, falls, sports injuries).
- Penetrating wounds from bullets or other objects.
- Birth injury.
- Toxic substances (lead, mercury, carbon monoxide).
- Infections (meningitis, encephalitis).
- Stroke or transient ischemic attack (TIA).
- Brain tumors.

3. Diagnosis of Brain Injury:
- Physical examination.
- Neurological exam.
- Imaging tests such as CT scan or MRI.
- Blood tests to rule out infections or toxic substances.
- Cerebrospina

**Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?(With all 5 combinations)**

In [ ]:
for config in prompt_configurations:

    print("=" * 100)
    print(config["name"])
    print("=" * 100)

    answer = generate_prompt_response(
        question=query_5.replace("[INST]", "").replace("[/INST]", ""),
        system_instruction=config["instruction"],
        max_tokens=config["max_tokens"],
        temperature=config["temperature"],
        top_p=config["top_p"],
        top_k=config["top_k"]
    )

    print(answer)

Configuration 1 - Basic


Llama.generate: prefix-match hit


A fractured leg, also known as a broken leg, requires immediate medical attention. Here are the necessary precautions and treatment steps:

1. Immediate Care: If you suspect a fracture, do not move the person unless it is necessary to prevent harm. Apply a makeshift splint if possible, using materials like sticks, clothes, or backpacking poles. Do not attempt to realign the bone.

2. Transportation: Call for emergency medical help and provide them with your location. If transporting the person yourself, try to keep them as comfortable as possible during the journey. Keep in mind that bumps and jolts can cause additional pain and discomfort.

3. Hospital Treatment: In the hospital, doctors will assess the fracture and determine the best course of treatment. This may include setting the bone with pins or a cast, administering pain medication, and monitoring for complications like infection or blood clots.

4. Recovery: The recovery process can take several weeks to months depending on th

Llama.generate: prefix-match hit


1. Key symptoms/findings: A person with a fractured leg may experience severe pain, swelling, bruising, deformity, or inability to bear weight on the affected limb. In some cases, there may be visible bone protrusion through the skin (open fracture), which requires immediate medical attention.

2. Possible causes or diagnosis: A leg fracture can occur due to various reasons such as falling, twisting, or direct trauma. The diagnosis is typically made based on a physical examination and imaging studies like X-rays.

3. Treatment/management: The initial step in managing a leg fracture is immobilization using a cast, splint, or traction to prevent movement and promote healing. Pain management through medications, ice packs, or elevation may also be necessary. In severe cases, surgery might be required to realign the bone and stabilize it with plates, screws, or rods. Rehabilitation including physical therapy and exercises will help restore strength and mobility.

4. Important precautions: 

...

A leg fracture, also known as a broken leg, is a common injury that can occur during physical activities like hiking. Here's a clinically organized answer to your question:

Symptoms:
- Severe pain in the affected leg
- Swelling and bruising around the injured area
- Difficulty walking or putting weight on the leg
- Deformity or misalignment of the bone, if it is a severe fracture
- Numbness or tingling sensations below the injury site

Diagnosis:
A healthcare professional will diagnose a leg fracture based on a physical examination and medical imaging tests such as X-rays.

Treatment:
1. Immediate first aid: Apply a splint or immobilize the leg with a makeshift sling to prevent further damage and provide support.
2. Seek medical attention: It's essential to seek professional medical care as soon as possible for proper diagnosis, treatment, and follow-up care.
3. Medical intervention: Depending on the severity of the fracture, treatments may include casting, surgery, or the use of a bo

Llama.generate: prefix-match hit


A leg fracture sustained during a hiking trip requires prompt medical attention to ensure proper healing and prevent potential complications. Here are the necessary precautions and treatment steps:

1. Assess the severity of the injury: Determine if it is an open or closed fracture, and assess for any signs of nerve or blood vessel damage, such as numbness, tingling, or pulselessness. If there are any signs of severe bleeding or shock, call emergency services immediately.
2. Immobilize the leg: Use a splint, sling, or a makeshift device to prevent further movement and potential worsening of the fracture. Be careful not to apply too much pressure on the injury site.
3. Provide pain relief: Over-the-counter pain medications like acetaminophen or ibuprofen can help manage pain. If the pain is severe, prescription pain medication may be necessary.
4. Seek medical attention: It's essential to see a healthcare professional as soon as possible for proper diagnosis and treatment. X-rays will b

Llama.generate: prefix-match hit


Title: Fractured Leg: Necessary Precautions, Treatment, and Recovery Considerations

1. Symptoms:
- Intense pain in the affected leg
- Swelling, bruising, or deformity around the fracture site
- Difficulty moving the leg or walking
- Inability to bear weight on the leg
- Possible numbness or tingling sensations

2. Possible Causes:
A fractured leg can occur due to various reasons such as falling, tripping, or twisting the leg in an unusual way while hiking.

3. Diagnosis:
The diagnosis of a fractured leg involves a thorough physical examination and medical history assessment. Imaging tests like X-rays are typically used to confirm the diagnosis and determine the severity and location of the fracture.

4. Treatment:
- RICE (Rest, Ice, Compression, Elevation): Rest the leg as much as possible, apply ice packs to reduce swelling, use a compression bandage or brace, and keep the leg elevated above heart level.
- Immobilization: A cast, splint, or brace may be used to immobilize the fractur

**Observation:**


The five configurations illustrate different levels of depth and clinical framing:

Basic gives a quick overview of the condition and highlights common treatments.

Structured organizes symptoms, causes, and management clearly, while also noting alternative causes.

Clinical provides a guideline‑style explanation, including diagnosis methods and treatment options.

Conservative emphasizes a cautious approach, listing multiple possible causes (autoimmune, stress, genetics, medications) and a range of treatments from topical to systemic.

Detailed offers the most comprehensive coverage, including symptoms, causes, diagnosis, treatment options, complications, and precautions.

**Overall insight:** Shorter outputs are useful for rapid recognition and quick decision‑support, while structured and detailed versions provide richer context for clinical validation, teaching, and patient communication.

##Data Preparation for RAG

###**Loading the Data**

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma



In [ ]:
pdf_path = "/content/medical_diagnosis_manual.pdf"

loader = PyMuPDFLoader(pdf_path)

pages = loader.load()

print("Number of pages loaded:", len(pages))

Number of pages loaded: 4114


###Data Overview

**Checking the first 5 pages**

In [ ]:
for i in range(min(5, len(pages))):

    print("=" * 100)
    print(f"PAGE {i + 1}")
    print("=" * 100)

    print(pages[i].page_content[:1500])

PAGE 1
shahanam04@gmail.com
UPWIKCV2H5
This file is meant for personal use by shahanam04@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.
PAGE 2
shahanam04@gmail.com
UPWIKCV2H5
This file is meant for personal use by shahanam04@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.
PAGE 3
Table of Contents
1
Front    ................................................................................................................................................................................................................
1
Cover    .......................................................................................................................................................................................................
2
Front Matter    ..............................................................................................................................................................

**Observation:**

 The PDF contains extensive medical information organized across multiple sections. Page-level metadata is retained so that retrieved information can later be traced back to the source document.

###**Checking the number of pages**

In [ ]:
print("Total number of pages:", len(pages))

Total number of pages: 4114


In [ ]:
print("Metadata of first page:")
print(pages[0].metadata)

Metadata of first page:
{'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'creator': 'Atop CHM to PDF Converter', 'creationdate': '2012-06-15T05:44:40+00:00', 'source': '/content/medical_diagnosis_manual.pdf', 'file_path': '/content/medical_diagnosis_manual.pdf', 'total_pages': 4114, 'format': 'PDF 1.7', 'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2026-08-11T04:47:29+00:00', 'trapped': '', 'modDate': 'D:20260811044729Z', 'creationDate': 'D:20120615054440Z', 'page': 0}


###Data Chunking

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = text_splitter.split_documents(pages)

print("Number of chunks:", len(chunks))

Number of chunks: 11972


Check

In [ ]:
print("First chunk:")
print(chunks[0].page_content[:1500])

print("\nMetadata:")
print(chunks[0].metadata)

First chunk:
shahanam04@gmail.com
UPWIKCV2H5
This file is meant for personal use by shahanam04@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.

Metadata:
{'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'creator': 'Atop CHM to PDF Converter', 'creationdate': '2012-06-15T05:44:40+00:00', 'source': '/content/medical_diagnosis_manual.pdf', 'file_path': '/content/medical_diagnosis_manual.pdf', 'total_pages': 4114, 'format': 'PDF 1.7', 'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2026-08-11T04:47:29+00:00', 'trapped': '', 'modDate': 'D:20260811044729Z', 'creationDate': 'D:20120615054440Z', 'page': 0}


**Observation:**

 The document is divided into overlapping chunks so that individual retrieved passages contain sufficient context while avoiding the need to pass the entire 4,000+ page manual to the LLM. A chunk overlap of 200 characters helps preserve contextual continuity between neighboring chunks.

###**Embedding**

In [ ]:
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings

embedding_model = SentenceTransformerEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully.")

/tmp/ipykernel_13388/3542779328.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = SentenceTransformerEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully.


**Observation:**

Sentence Transformer embeddings convert the text chunks into numerical vectors representing their semantic meaning. This allows the retriever to identify passages that are semantically related to the user's question rather than relying only on exact keyword matching.

###**Vector Database**

In [ ]:
from langchain_community.vectorstores import Chroma

vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_name="merck_medical_manual"
)

print("Vector database created successfully.")

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Vector database created successfully.


**Observation:**

The medical document chunks and their embeddings are stored in a Chroma vector database. This provides the retrieval layer required by the RAG architecture.

###**Retriever**

In [ ]:
retriever = vector_db.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

print("Retriever configured successfully.")
print("Search type: similarity")
print("k:", 3)

Retriever configured successfully.
Search type: similarity
k: 3


In [ ]:
test_query = "What are the symptoms and treatment of sepsis?"

retrieved_docs = retriever.invoke(test_query)

print("Retrieved documents:", len(retrieved_docs))

for i, doc in enumerate(retrieved_docs, 1):

    print("=" * 100)
    print(f"Retrieved Chunk {i}")
    print("Metadata:", doc.metadata)
    print(doc.page_content[:1000])

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Retrieved documents: 3
Retrieved Chunk 1
Metadata: {'author': '', 'creationDate': 'D:20120615054440Z', 'creationdate': '2012-06-15T05:44:40+00:00', 'creator': 'Atop CHM to PDF Converter', 'file_path': '/content/medical_diagnosis_manual.pdf', 'format': 'PDF 1.7', 'keywords': '', 'modDate': 'D:20260811044729Z', 'moddate': '2026-08-11T04:47:29+00:00', 'page': 2454, 'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'source': '/content/medical_diagnosis_manual.pdf', 'subject': '', 'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition', 'total_pages': 4114, 'trapped': ''}
factors, excessive fibrinolysis in reaction thereto, and more often a combination of both.
Symptoms and Signs
With sepsis, the patient typically has fever, tachycardia, and tachypnea; BP remains
[Table 227-1. Sepsis in the United States]
normal. Other signs of the causative infection are generally present. As severe sepsis or septic shock
develops, the first sign may be confusion or decreased alertness. 

**Observation:**

 Similarity-based retrieval with k=3 returns the three document chunks that are most semantically similar to the query. This provides the LLM with focused medical context instead of the complete manual.

###**System and User Prompt Template**

In [ ]:
qna_system_message = """
You are a medical information assistant supporting healthcare professionals.

Use ONLY the information contained in the provided context from the Merck Manuals
to answer the question.

Do not invent facts or introduce unsupported medical information.

If the provided context does not contain enough information to answer the question,
clearly state that the available context is insufficient.

Present the response in a clear and clinically organized manner.

Include:
- Key symptoms/findings
- Possible causes or diagnosis
- Treatment/management
- Important precautions
- Relevant source information

This system is intended for medical information support and does not replace
professional clinical judgment.
"""

qna_user_message_template = """
Context from the Merck Manuals:

{context}

Question:

{question}

Answer the question using only the provided context.
"""

**Response Function**

In [ ]:
def generate_rag_response(
    user_input,
    k=3,
    max_tokens=300,
    temperature=0,
    top_p=0.95,
    top_k=50
):

    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.invoke(
        user_input
    )[:k]

    context_list = [
        d.page_content
        for d in relevant_document_chunks
    ]

    # Combine retrieved chunks
    context_for_query = "\n\n".join(context_list)

    user_message = qna_user_message_template.replace(
        "{context}",
        context_for_query
    )

    user_message = user_message.replace(
        "{question}",
        user_input
    )

    prompt = f"""[INST]
{qna_system_message}

{user_message}
[/INST]"""

    try:

        model_output = llm(
            prompt=prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k
        )

        generated_response = model_output[
            "choices"
        ][0]["text"].strip()

    except Exception as e:

        generated_response = (
            f"Sorry, I encountered the following error:\n{e}"
        )

    return generated_response, relevant_document_chunks

##**Question Answering using RAG**

`Query 1: What is the protocol for managing sepsis in a critical care unit?`

In [ ]:
answer, retrieved_docs = generate_rag_response(
    query_1.replace("[INST]", "").replace("[/INST]", ""),
    k=3,
    max_tokens=350,
    temperature=0,
    top_p=0.95,
    top_k=50
)

print("RAG ANSWER:")
print(answer)

print("\nRETRIEVED SOURCES:")

for i, doc in enumerate(retrieved_docs, 1):

    print(
        f"\nSource {i} | "
        f"Page: {doc.metadata.get('page', 'N/A')}"
    )

    print(doc.page_content[:500])

Llama.generate: prefix-match hit


RAG ANSWER:
The protocol for managing sepsis in a critical care unit includes the following steps:

1. Fluid resuscitation with 0.9% normal saline until central venous pressure (CVP) reaches 8 mm Hg (10 cm H2O) or pulmonary artery occlusive pressure (PAOP) reaches 12 to 15 mm Hg. Oliguria with hypotension is not a contraindication to vigorous fluid resuscitation.
2. Administration of oxygen.
3. Broad-spectrum antibiotics, modified by culture results.
4. Drainage of abscesses and excision of necrotic tissue.
5. Normalization of blood glucose levels.
6. Replacement-dose corticosteroids (24 μg/kg/h by continuous IV infusion for 96 hours). However, contraindications include hemorrhagic stroke within 3 months, spinal or intracranial surgery within 2 months, acute trauma with a risk of bleeding, and intracranial neoplasm.
7. Monitoring: systemic pressure, CVP or PAOP, pulse oximetry, arterial blood gases (ABGs), blood glucose, lactate, and electrolyte levels, renal function, sublingual PCO2,

`Query 2: What are the common symptoms of appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?`

In [ ]:
answer, retrieved_docs = generate_rag_response(
    query_2.replace("[INST]", "").replace("[/INST]", ""),
    k=3,
    max_tokens=350,
    temperature=0,
    top_p=0.95,
    top_k=50
)

print("RAG ANSWER:")
print(answer)

print("\nRETRIEVED SOURCES:")

for i, doc in enumerate(retrieved_docs, 1):

    print(
        f"\nSource {i} | "
        f"Page: {doc.metadata.get('page', 'N/A')}"
    )

    print(doc.page_content[:500])

Llama.generate: prefix-match hit


RAG ANSWER:
The common symptoms for appendicitis include epigastric or periumbilical pain followed by brief nausea, vomiting, and anorexia, which is later followed by pain shifting to the right lower quadrant. Pain increases with cough and motion. Classical signs are direct and rebound tenderness at McBurney's point, Rovsing sign, psoas sign, or obturator sign. A low-grade fever (rectal temperature 100 to 101°F) is common. Unfortunately, these classic findings appear in less than 50% of patients, and many variations of symptoms and signs occur. Appendicitis is typically treated with surgical removal, and if a perforation is noted, immediate surgery is necessary due to the risk of mortality from peritonitis increasing rapidly with delayed treatment. If an abscess or inflammatory mass has formed, the procedure may be limited to drainage of the abscess. Antibiotics and IV fluids are also administered for treatment. The surgical procedure followed to treat appendicitis is open or laparosco

`Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?`

In [ ]:
answer, retrieved_docs = generate_rag_response(
    query_3.replace("[INST]", "").replace("[/INST]", ""),
    k=3,
    max_tokens=350,
    temperature=0,
    top_p=0.95,
    top_k=50
)

print("RAG ANSWER:")
print(answer)

print("\nRETRIEVED SOURCES:")

for i, doc in enumerate(retrieved_docs, 1):

    print(
        f"\nSource {i} | "
        f"Page: {doc.metadata.get('page', 'N/A')}"
    )

    print(doc.page_content[:500])

Llama.generate: prefix-match hit


RAG ANSWER:
The context from the Merck Manuals indicates that sudden patchy hair loss, also known as alopecia areata, is a condition characterized by bald spots on the scalp or other hair-bearing areas with no obvious skin or systemic disorder. The causes of this condition are believed to be autoimmune in nature and may affect genetically susceptible individuals exposed to unclear environmental triggers.

Effective treatments for alopecia areata include:

1. Topical corticosteroids: These medications can help reduce inflammation and promote hair regrowth. They can be applied directly to the affected areas in various forms, such as foams, lotions, or solutions.
2. Intralesional corticosteroid injections: This treatment involves injecting corticosteroids directly into the affected area using a fine needle. It may provide more effective results than topical applications but can cause more side effects.
3. Topical minoxidil: This medication is applied to the scalp twice daily and has been 

`Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?`

In [ ]:
answer, retrieved_docs = generate_rag_response(
    query_4.replace("[INST]", "").replace("[/INST]", ""),
    k=3,
    max_tokens=350,
    temperature=0,
    top_p=0.95,
    top_k=50
)

print("RAG ANSWER:")
print(answer)

print("\nRETRIEVED SOURCES:")

for i, doc in enumerate(retrieved_docs, 1):

    print(
        f"\nSource {i} | "
        f"Page: {doc.metadata.get('page', 'N/A')}"
    )

    print(doc.page_content[:500])

Llama.generate: prefix-match hit


RAG ANSWER:
Based on the information from the Merck Manual, the following treatments are recommended for a person who has sustained a physical injury to brain tissue:

1. For mild injuries, discharge and observation are sufficient.
2. For moderate and severe injuries, optimization of ventilation, oxygenation, and brain perfusion is necessary. Treatment of complications such as increased intracranial pressure (ICP), seizures, and hematomas is also essential. Rehabilitation is recommended.
3. Initial resuscitation of trauma patients includes securing a clear airway, controlling external bleeding, avoiding displacement of the spine or other bones, and providing pain relief with a short-acting opioid.
4. For larger lesions, rapidly developing lesions, and lesions that affect both hemispheres, treatment focuses on managing symptoms and supporting overall health to promote recovery.
5. Recovery from brain injury depends on the plasticity of the remaining cerebrum, which varies from person to

`Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?`

In [ ]:
answer, retrieved_docs = generate_rag_response(
    query_5.replace("[INST]", "").replace("[/INST]", ""),
    k=3,
    max_tokens=350,
    temperature=0,
    top_p=0.95,
    top_k=50
)

print("RAG ANSWER:")
print(answer)

print("\nRETRIEVED SOURCES:")

for i, doc in enumerate(retrieved_docs, 1):

    print(
        f"\nSource {i} | "
        f"Page: {doc.metadata.get('page', 'N/A')}"
    )

    print(doc.page_content[:500])

Llama.generate: prefix-match hit


RAG ANSWER:
Based on the information from the Merck Manuals, a person with a fractured leg (femoral shaft fracture) from a hiking incident should receive the following care:

1. Initial assessment and stabilization: In case of severe pain, swelling, deformity, or instability, immediate splinting is necessary to prevent further injury and decrease pain. Hemorrhagic shock, if present, should be treated in the emergency department.

2. Definitive treatment: ORIF (Open Reduction Internal Fixation) is typically used for femoral shaft fractures. This procedure involves surgically aligning and stabilizing the bone with plates or screws.

3. Rehabilitation: After surgery, rehabilitation should be initiated as soon as possible to increase strength, prevent atrophy on the unaffected side, and promote full ambulation. Initially, only isometric exercises are permitted while the affected limb is fully extended. Gradual mobilization will eventually lead to full ambulation.

4. Precautions: For patie

**Observation:**

The generated response integrates this evidence into a structured treatment overview while noting variability in patient outcomes.

**Overall insight:** All the outputs shows the strength of RAG in medical decision-support:

It reduces information overload by surfacing the most relevant passages.

It improves traceability by exposing page references.

It balances concise AI-generated summaries with authoritative source material, ensuring clinicians can verify and contextualize recommendations.

In short, RAG makes the output more reliable, transparent, and clinically useful than a standalone LLM response.

###**Fine-tuning**

Fine-tune the retriever, and LLM parameters (at least 5 combinations) to check different results.

In [ ]:
rag_configurations = [
    {
        "name": "RAG Configuration 1",
        "k": 2,
        "temperature": 0,
        "top_p": 0.90,
        "top_k": 40,
        "max_tokens": 300
    },

    {
        "name": "RAG Configuration 2",
        "k": 3,
        "temperature": 0,
        "top_p": 0.95,
        "top_k": 50,
        "max_tokens": 350
    },

    {
        "name": "RAG Configuration 3",
        "k": 4,
        "temperature": 0.1,
        "top_p": 0.90,
        "top_k": 40,
        "max_tokens": 400
    },

    {
        "name": "RAG Configuration 4",
        "k": 5,
        "temperature": 0.2,
        "top_p": 0.90,
        "top_k": 50,
        "max_tokens": 450
    },

    {
        "name": "RAG Configuration 5",
        "k": 6,
        "temperature": 0,
        "top_p": 0.85,
        "top_k": 30,
        "max_tokens": 500
    }
]

In [ ]:
rag_experiment_results = []

clean_questions = [
    query_1.replace("[INST]", "").replace("[/INST]", ""),
    query_2.replace("[INST]", "").replace("[/INST]", ""),
    query_3.replace("[INST]", "").replace("[/INST]", ""),
    query_4.replace("[INST]", "").replace("[/INST]", ""),
    query_5.replace("[INST]", "").replace("[/INST]", "")
]

for config in rag_configurations:

    for question_number, question in enumerate(
        clean_questions,
        1
    ):

        answer, retrieved_docs = generate_rag_response(
            question,
            k=config["k"],
            max_tokens=config["max_tokens"],
            temperature=config["temperature"],
            top_p=config["top_p"],
            top_k=config["top_k"]
        )

        rag_experiment_results.append({
            "Configuration": config["name"],
            "Question": question_number,
            "K": config["k"],
            "Temperature": config["temperature"],
            "Top_P": config["top_p"],
            "Top_K": config["top_k"],
            "Answer": answer
        })

rag_results_df = pd.DataFrame(
    rag_experiment_results
)

rag_results_df

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


,Configuration,Question,K,Temperature,Top_P,Top_K,Answer
0,RAG Configuration 1,1,2,0.0,0.90,40,The protocol for managing sepsis in a critical...
1,RAG Configuration 1,2,2,0.0,0.90,40,The common symptoms for appendicitis include e...
2,RAG Configuration 1,3,2,0.0,0.90,40,The context from the Merck Manuals indicates t...
3,RAG Configuration 1,4,2,0.0,0.90,40,Based on the information from the Merck Manual...
4,RAG Configuration 1,5,2,0.0,0.90,40,Based on the information from the Merck Manual...
5,RAG Configuration 2,1,3,0.0,0.95,50,The protocol for managing sepsis in a critical...
6,RAG Configuration 2,2,3,0.0,0.95,50,The common symptoms for appendicitis include e...
7,RAG Configuration 2,3,3,0.0,0.95,50,The context from the Merck Manuals indicates t...
8,RAG Configuration 2,4,3,0.0,0.95,50,Based on the information from the Merck Manual...
9,RAG Configuration 2,5,3,0.0,0.95,50,Based on the information from the Merck Manual...


**Observations:**

The RAG experiments demonstrate that both retrieval and generation parameters influence answer quality. A lower k produces a more focused context but may omit information when the answer spans multiple sections.

Increasing k can improve information coverage but may also introduce irrelevant passages. Lower temperature values generally produce more deterministic responses, which is preferable for medical information retrieval. The final configuration should be selected based on the groundedness and relevance evaluation rather than response length alone.

###**Output Evaluation**

We are using the same Mistral model for evaluation with improved prompts.

**Groundedness evaluator**

In [ ]:
groundedness_rater_system_message = """
You are an evaluator for a medical Retrieval-Augmented Generation system.

Your task is to evaluate the GROUNDEDNESS of the generated answer.

Groundedness means whether the claims made in the answer are supported
by the retrieved context from the Merck Manuals.

Use this scoring scale:

5 = Fully grounded. All important claims are supported by the context.
4 = Mostly grounded. Most claims are supported, with minor unsupported details.
3 = Partially grounded. Some important claims are supported, but others are not.
2 = Mostly ungrounded. Only a small portion is supported.
1 = Ungrounded. The answer is not supported by the provided context.

Do not judge whether the answer is medically correct in general.
Judge only whether it is supported by the supplied context.

Return:
Score: <1-5>
Reason: <brief explanation>
"""

**Relevance evaluator**

In [ ]:
relevance_rater_system_message = """
You are an evaluator for a medical question-answering system.

Your task is to evaluate the RELEVANCE of the generated answer.

Relevance means whether the answer directly addresses the user's question
and covers the major aspects requested.

Use this scoring scale:

5 = Fully relevant and complete.
4 = Mostly relevant with minor omissions.
3 = Partially relevant; some important aspects are missing.
2 = Mostly irrelevant or incomplete.
1 = Completely irrelevant.

Return:
Score: <1-5>
Reason: <brief explanation>
"""

**User message template**

In [ ]:
user_message_template = """
QUESTION:
{question}

RETRIEVED CONTEXT:
{context}

GENERATED ANSWER:
{answer}

Evaluate the generated answer according to your assigned evaluation criterion.
"""

**Evaluation function**

In [ ]:
def generate_ground_relevance_response(
    user_input,
    k=3,
    max_tokens=200,
    temperature=0,
    top_p=0.95,
    top_k=50
):

    # Retrieve documents
    relevant_document_chunks = retriever.invoke(
        user_input
    )[:k]

    context_for_query = "\n\n".join(
        [d.page_content for d in relevant_document_chunks]
    )

    # Generate RAG answer
    qna_prompt = f"""[INST]
{qna_system_message}

{qna_user_message_template.format(
    context=context_for_query,
    question=user_input
)}
[/INST]"""

    answer_response = llm(
        prompt=qna_prompt,
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k
    )

    answer = answer_response[
        "choices"
    ][0]["text"].strip()

    # Groundedness evaluation
    groundedness_prompt = f"""[INST]
{groundedness_rater_system_message}

{user_message_template.format(
    context=context_for_query,
    question=user_input,
    answer=answer
)}
[/INST]"""

    groundedness_response = llm(
        prompt=groundedness_prompt,
        max_tokens=200,
        temperature=0,
        top_p=0.95,
        top_k=50
    )

    # Relevance evaluation
    relevance_prompt = f"""[INST]
{relevance_rater_system_message}

{user_message_template.format(
    context=context_for_query,
    question=user_input,
    answer=answer
)}
[/INST]"""

    relevance_response = llm(
        prompt=relevance_prompt,
        max_tokens=200,
        temperature=0,
        top_p=0.95,
        top_k=50
    )

    groundedness = groundedness_response[
        "choices"
    ][0]["text"].strip()

    relevance = relevance_response[
        "choices"
    ][0]["text"].strip()

    return answer, groundedness, relevance

###**Evaluation Queries**

`Query 1: What is the protocol for managing sepsis in a critical care unit?`

In [ ]:
answer, groundedness, relevance = generate_ground_relevance_response(
    clean_questions[0],
    k=3
)

print("ANSWER")
print(answer)

print("\nGROUNDEDNESS EVALUATION")
print(groundedness)

print("\nRELEVANCE EVALUATION")
print(relevance)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


ANSWER
The protocol for managing sepsis in a critical care unit includes the following steps:

1. Fluid resuscitation with 0.9% normal saline until central venous pressure (CVP) reaches 8 mm Hg (10 cm H2O) or pulmonary artery occlusive pressure (PAOP) reaches 12 to 15 mm Hg. Oliguria with hypotension is not a contraindication to vigorous fluid resuscitation.
2. Administration of oxygen.
3. Broad-spectrum antibiotics, modified by culture results.
4. Drainage of abscesses and excision of necrotic tissue.
5. Normalization of blood glucose levels.
6. Replacement-dose corticosteroids (24 μg/kg/h by continuous IV infusion for 96 hours). However, contraind

GROUNDEDNESS EVALUATION
Score: 5
Reason: The generated answer fully matches the retrieved context in terms of the steps involved in managing sepsis in a critical care unit. All important claims are supported by the context.

RELEVANCE EVALUATION
Score: 5
Reason: The generated answer directly addresses the user's question and covers all the

`Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?`

In [ ]:
answer, groundedness, relevance = generate_ground_relevance_response(
    clean_questions[1],
    k=3
)

print("ANSWER")
print(answer)

print("\nGROUNDEDNESS EVALUATION")
print(groundedness)

print("\nRELEVANCE EVALUATION")
print(relevance)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


ANSWER
The common symptoms for appendicitis include epigastric or periumbilical pain followed by brief nausea, vomiting, and anorexia, which is then followed by pain shifting to the right lower quadrant. Pain increases with cough and motion. Classic signs are direct and rebound tenderness located at McBurney's point, Rovsing sign, psoas sign, or obturator sign. A low-grade fever (rectal temperature 100 to 101°F) is common. Unfortunately, these classic findings appear in less than 50% of patients, and many variations of symptoms and signs occur. Appendicitis is typically treated with surgical removal, specifically open or laparoscopic appendectomy. If a perforation is noted, immediate surgery is necessary due to the risk of mortality from peritonitis increasing rapidly with delayed treatment. Antibiotics and IV fluids

GROUNDEDNESS EVALUATION
Score: 5
Reason: The generated answer accurately and fully conveys the common symptoms for appendicitis as described in the context, including epi

`Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?`

In [ ]:
answer, groundedness, relevance = generate_ground_relevance_response(
    clean_questions[2],
    k=3
)

print("ANSWER")
print(answer)

print("\nGROUNDEDNESS EVALUATION")
print(groundedness)

print("\nRELEVANCE EVALUATION")
print(relevance)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


ANSWER
The context from the Merck Manuals indicates that sudden patchy hair loss, also known as alopecia areata, is a condition characterized by bald spots on the scalp or any other hair-bearing area. This condition affects people with no obvious skin or systemic disorder. Alopecia areata is believed to be an autoimmune disorder and may be triggered by unclear environmental factors.

The effective treatments for addressing sudden patchy hair loss include:

1. Topical corticosteroids: These medications can help reduce inflammation and promote hair regrowth. They can be applied directly to the affected area.
2. Intralesional corticosteroid injections: Corticosteroids can also be injected into the affected area for more effective treatment.
3. Topical minoxidil: This medication can stimulate hair growth and may be applied twice daily to the affected area.
4. Topical anth

GROUNDEDNESS EVALUATION
Score: 5
Reason: The generated answer accurately summarizes the information from the context r

`Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?`

In [ ]:
answer, groundedness, relevance = generate_ground_relevance_response(
    clean_questions[3],
    k=3
)

print("ANSWER")
print(answer)

print("\nGROUNDEDNESS EVALUATION")
print(groundedness)

print("\nRELEVANCE EVALUATION")
print(relevance)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


ANSWER
Based on the information from the Merck Manual, the following treatments are recommended for a person who has sustained a physical injury to brain tissue:

1. For mild injuries, discharge and observation are sufficient.
2. For moderate and severe injuries, optimization of ventilation, oxygenation, and brain perfusion is necessary. Treatment of complications such as increased intracranial pressure (ICP), seizures, and hematomas is also essential. Rehabilitation is recommended.
3. Initial resuscitation of trauma patients includes securing a clear airway, controlling external bleeding, avoiding displacement of the spine or other bones, and providing pain relief with a short-acting opioid.
4. For larger lesions, rapidly developing lesions, and lesions that affect both hemispheres, treatment focuses on managing symptoms and supporting overall health to promote recovery.
5. Recovery from brain injury depends on the plasticity of the

GROUNDEDNESS EVALUATION
Score: 5
Reason: The genera

`Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?`

In [ ]:
answer, groundedness, relevance = generate_ground_relevance_response(
    clean_questions[4],
    k=3
)

print("ANSWER")
print(answer)

print("\nGROUNDEDNESS EVALUATION")
print(groundedness)

print("\nRELEVANCE EVALUATION")
print(relevance)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


ANSWER
Based on the information from the Merck Manuals, a person with a fractured leg (femoral shaft fracture) from a hiking incident should receive the following care:

1. Initial assessment and stabilization: In case of severe pain, swelling, deformity, or instability, immediate splinting is necessary to prevent further injury and decrease pain. Hemorrhagic shock, if present, should be treated in the emergency department with fluid resuscitation and surgical repair for life- or limb-threatening injuries.

2. Definitive treatment: Open reduction with internal fixation (ORIF) is typically used to realign the fractured bone. Early mobilization is essential for optimal recovery.

3. Precautions during transport: If necessary, the person should be transported carefully to ensure that the fracture remains stable and does not cause further harm.

4. Rehabilitation: After surgery,

GROUNDEDNESS EVALUATION
Score: 5
Reason: The generated answer fully grounds all important claims with the conte

**Observation:**

The RAG answer shows strong alignment between retrieved Merck Manual content and the generated response. It clearly outlines initial stabilization, definitive treatment, transport precautions and rehabilitation steps.

The groundedness score of 5 reflects that all claims are supported by authoritative source material, while the relevance score of 5 confirms the response directly addresses the clinical scenario posed.

**Overall insight:** This example highlights how RAG can deliver clinically accurate, traceable, and context‑specific guidance. It balances concise decision‑support with transparency, ensuring healthcare professionals can verify the recommendations against trusted references.

###**consolidated evaluation table to make notebook much stronger**

In [ ]:
evaluation_results = []

for i, question in enumerate(clean_questions, 1):

    answer, groundedness, relevance = (
        generate_ground_relevance_response(
            question,
            k=3
        )
    )

    evaluation_results.append({
        "Question": i,
        "Answer": answer,
        "Groundedness Evaluation": groundedness,
        "Relevance Evaluation": relevance
    })

evaluation_df = pd.DataFrame(
    evaluation_results
)

evaluation_df

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


,Question,Answer,Groundedness Evaluation,Relevance Evaluation
0,1,The protocol for managing sepsis in a critical...,Score: 5\nReason: The generated answer fully m...,Score: 5\nReason: The generated answer directl...
1,2,The common symptoms for appendicitis include e...,Score: 5\nReason: The generated answer accurat...,Score: 5\nReason: The generated answer fully a...
2,3,The context from the Merck Manuals indicates t...,Score: 5\nReason: The generated answer accurat...,Score: 5\nReason: The generated answer directl...
3,4,Based on the information from the Merck Manual...,Score: 5\nReason: The generated answer fully g...,Score: 5\nReason: The generated answer directl...
4,5,Based on the information from the Merck Manual...,Score: 5\nReason: The generated answer fully g...,Score: 5\nReason: The generated answer directl...


**Observations**

**Groundedness:** The groundedness evaluation measures whether the generated answer is supported by the retrieved Merck Manual context. Higher scores indicate that the RAG pipeline is successfully restricting the answer to information retrieved from the knowledge base. Lower scores indicate either retrieval problems or generation of information that was not present in the retrieved context.

**Relevance:** The relevance evaluation measures whether the generated answer addresses the user's question. A high relevance score indicates that the retrieved context and generation process are collectively addressing the requested medical topic.

**Overall:** The evaluation demonstrates the importance of both retrieval quality and generation quality. A response can be relevant but insufficiently grounded if the model introduces information beyond the retrieved context. Conversely, an answer can be well grounded but incomplete if the retriever fails to retrieve all the information required to answer the question.

##**Actionable Insights and Business Recommendations**

**Reduce information overload**  
Semantic retrieval from a large medical knowledge base minimizes the need for clinicians to manually search thousands of pages, streamlining access to critical information.

**Enhance decision‑support efficiency**  
Relevant passages can be retrieved in seconds and paired with concise AI‑generated summaries, cutting down the time required to locate and interpret reference material.

**Improve transparency and traceability**  
Unlike standalone LLMs, the RAG system exposes the retrieved Merck Manual passages and page metadata. This allows healthcare professionals to verify the source material behind each response.

**Standardize access to medical knowledge**  
A centralized retrieval interface ensures consistent access to approved medical resources, supporting standardized practices across healthcare teams.

**Optimize retrieval and generation parameters**  
Experiments show that parameters such as the number of retrieved chunks (k), temperature, top‑p, and top‑k directly affect response quality. Configurations should be tuned for groundedness and relevance, not just response length.

**Maintain human oversight**  
The system is positioned as a decision‑support tool, not an autonomous diagnostic engine. Clinicians must validate AI‑generated insights against authoritative sources and patient‑specific circumstances.

**Keep the knowledge base current**  
The vector database should be refreshed whenever medical manuals or approved resources are updated, ensuring retrieval reflects the latest information without retraining the language model.

**Focus on information retrieval as the core use case**  
The strongest business value lies in rapid retrieval and summarization of trusted medical knowledge, helping clinicians manage information overload while maintaining a clear link to the underlying reference material.

###**Overall Conclusion:**

The project demonstrates the progression from a standalone LLM to a prompt-engineered LLM and finally to a Retrieval-Augmented Generation system using the Merck Manuals as the domain knowledge source.

The baseline LLM can generate medically relevant responses, but its answers lack source traceability. Prompt engineering improves structure, consistency and completeness.

The RAG approach further improves the system by retrieving relevant medical information from the supplied knowledge base before generating the answer. Groundedness and relevance evaluation provide a mechanism for comparing configurations and identifying potential weaknesses in retrieval or generation.

The proposed solution can therefore serve as a prototype for a healthcare knowledge-assistance platform, provided that appropriate human oversight, source validation, privacy controls and clinical governance are maintained.

**Thank You!**

In [ ]:
!jupyter nbconvert --to html Natural_Language_Processing_with_Generative_AI_Project.ipynb

[NbConvertApp] Converting notebook Natural_Language_Processing_with_Generative_AI_Project.ipynb to html
[NbConvertApp] Writing 691326 bytes to Natural_Language_Processing_with_Generative_AI_Project.html
